# 02 · Label Taxonomy

**Purpose.** Decide, with evidence, which six classes the model predicts and which source labels feed each class.

| | |
|---|---|
| **Reads** | raw metadata, `src/dermalens/labels.py` |
| **Writes** | `reports/dataset_audit/label_candidates.csv`, `label_mapping.csv` |
| **Run time (Colab T4)** | <1 min |
| **Next notebook** | 03 · Build Clean Manifest |

Run the cells from top to bottom. All shared logic lives in `src/dermalens/`; this
notebook only calls it and explains each decision.

## 0. Setup

The same cell starts every notebook. It (1) connects Google Drive on Colab,
(2) installs missing libraries on Colab, (3) makes `src/` importable,
(4) finds every project folder and (5) loads `configs/baseline.yaml`.

In [ ]:
import os
import sys
from pathlib import Path

# 1) On Colab, connect Google Drive so the project folder is visible.
try:
    from google.colab import drive                      # this import only works on Colab
    drive.mount("/content/drive")
    os.environ.setdefault("DERMALENS_PROJECT_DIR", "/content/drive/MyDrive/DermaLens_AI")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False                                    # local run: set DERMALENS_PROJECT_DIR yourself

# 2) Where the data lives (PROJECT_DIR) and where the code lives (CODE_DIR, normally the same folder).
PROJECT_DIR = Path(os.environ.get("DERMALENS_PROJECT_DIR", Path.cwd().parent))
CODE_DIR = Path(os.environ.get("DERMALENS_CODE_DIR", PROJECT_DIR))

# 3) Colab starts empty every session: install the few libraries it does not ship with.
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE_DIR / "requirements.txt")], check=True)

# 4) Make `import dermalens` work.
sys.path.insert(0, str(CODE_DIR / "src"))

from dermalens.config import load_config
from dermalens.paths import ProjectPaths
from dermalens.utils import set_seed

# 5) Folders and settings.
paths = ProjectPaths(PROJECT_DIR).ensure()
CONFIG = load_config(os.environ.get("DERMALENS_CONFIG", CODE_DIR / "configs" / "baseline.yaml"))
set_seed(CONFIG["seed"])
print(paths.describe())

## 1. Selection criteria

A disease becomes a target class when it is:
1. **common in Indian primary care** (infectious and inflammatory conditions, not cancer),
2. **well represented in DermaCon-IN** (at least ~140 images, so the test set has enough examples),
3. ideally **also present in SkinDisNet**, to add South Asian smartphone photos.

Healthy skin is the sixth class because a screening tool must be able to say "nothing found".

In [ ]:
import pandas as pd

derma = pd.read_csv(paths.raw / "dermaconin" / "METADATA" / "Skin_Metadata-1.csv")
skin = pd.read_csv(paths.raw / "skindisnet" / "skindisnet2" / "SkinDisNet_Metadata.csv")
derma_counts = derma["Disease_label"].str.strip().value_counts()
skin_counts = skin["Diagnosis"].str.strip().value_counts()

# The 12 most common DermaCon-IN diagnoses, side by side with SkinDisNet.
candidates = pd.DataFrame({
    "dermaconin_images": derma_counts.head(12),
    "dermaconin_subjects": derma.groupby(derma["Disease_label"].str.strip())["Subject_ID"].nunique(),
    "skindisnet_images": skin_counts,
}).dropna(subset=["dermaconin_images"]).fillna(0).astype(int).sort_values("dermaconin_images", ascending=False)
candidates.to_csv(paths.audit / "label_candidates.csv")
display(candidates)

## 2. The decision

| Class | Why it is in | Sources |
|---|---|---|
| Tinea Corporis | most common fungal infection; 323 DermaCon-IN + 275 SkinDisNet | both |
| Scabies | common parasitic infection | both |
| Eczema | common inflammatory condition | both |
| Contact Dermatitis | common; frequently confused with eczema (a hard, useful pair) | both |
| Psoriasis | common chronic condition, 159 images | DermaCon-IN only |
| Healthy | the "nothing found" answer | MCSI only |

**Not chosen:** Vitiligo and Tinea Cruris are larger but would change the project's scope (pigment
disorder; groin-site tinea). Acne needs different photos (face close-ups). These are listed in
`CANDIDATE_MERGES` in `labels.py` for discussion with the supervisor, but are **not active**.

The mapping below is read from `src/dermalens/labels.py`, the single place where it is defined.

In [ ]:
from dermalens.labels import CANDIDATE_MERGES, SOURCE_LABEL_MAP, TARGET_CLASSES

source_counts = {"dermaconin": derma_counts, "skindisnet": skin_counts}
rows = []
for source, mapping in SOURCE_LABEL_MAP.items():
    for source_label, target in mapping.items():
        images = source_counts.get(source, pd.Series(dtype=int)).get(source_label)
        rows.append({"source": source, "source_label": source_label, "target_class": target,
                     "images_in_source": images})
mapping_table = pd.DataFrame(rows)
mapping_table.to_csv(paths.audit / "label_mapping.csv", index=False)
display(mapping_table)
print("Target classes (model output order):", TARGET_CLASSES)

## 3. How sure were the dermatologists? (DermaCon-IN `Confidence`, 1 = unsure, 5 = certain)

In [ ]:
chosen = derma[derma["Disease_label"].str.strip().isin(SOURCE_LABEL_MAP["dermaconin"])]
confidence = pd.crosstab(chosen["Disease_label"].str.strip(), chosen["Confidence"])
display(confidence)
print(f"Share of chosen images labelled with confidence >= 4: {(chosen['Confidence'] >= 4).mean():.0%}")

## 4. What merging the candidate labels would add (not applied)

In [ ]:
for source, merges in CANDIDATE_MERGES.items():
    for source_label, idea in merges.items():
        print(f"{source}: '{source_label}' -> {idea}: +{int(derma_counts.get(source_label, 0))} images")

**Done.** The taxonomy is fixed in `labels.py`; notebook 03 applies it.